# Tamper Test
Tamper OrbitDB records via `/audit/tamper` programmatically, then verify on dashboard.

In [ ]:
SESSION_ID = "" # copy the session id on dashboard and paste it here.

1. Define helper functions:

In [ ]:
import requests

CONTROLLER_URL  = "http://localhost:3000"


def ctrl(path, method="GET", **kwargs):
    fn = getattr(requests, method.lower())
    timeout = kwargs.pop("timeout", 30)
    r = fn(f"{CONTROLLER_URL}{path}", timeout=timeout, **kwargs)
    r.raise_for_status()
    return r.json()

def tamper(entry_id, tamper_type, fake_cid=None):
    body = {"entry_id": entry_id, "tamper_type": tamper_type}
    if fake_cid:
        body["fake_cid"] = fake_cid
    return ctrl("/debug/tamper", method="POST", json=body)

def get_entry(entry_id):
    return ctrl(f"/debug/entry/{entry_id}")

def restore(entry_id, original_entry):
    return ctrl("/debug/restore", method="POST",
                json={"entry_id": entry_id, "original_entry": original_entry})

def derive_entry_id(session_id, ptype, action_or_id):
    """Mirror of deriveEntryId() in strings.js"""
    if ptype == 'session':
        return f"{session_id}-{action_or_id}"
    elif ptype in ('epoch', 'step'):
        return f"{session_id}-{ptype}-{action_or_id}"
    raise ValueError(f"Unknown packet type: {ptype}")

2. Get original entry:

In [ ]:
packets = ctrl(f"/audit/session/{SESSION_ID}/packets", timeout=300)
steps   = packets["steps"]
sample_step  = steps[len(steps) // 2]    # middle step
step_entry_id  = derive_entry_id(SESSION_ID, 'step', sample_step['step_id'])
original = get_entry(step_entry_id)

3. Tamper the entry:

In [ ]:
tamper(step_entry_id, "step_loss")

4. Restore the entry:

In [ ]:
restore(step_entry_id, original)